# Bonus: Extended Angle Encoding with $R_X$, $R_Y$, and $R_Z$

So far, we have only used $R_Y$ rotations for Angle Encoding. But the Bloch sphere has three axes! Here we show you how to meaningfully combine all three rotation axes.

**Learning Objectives:**
- Understand the effect of $R_X$, $R_Y$, and $R_Z$ on the Bloch sphere
- Implement multi-axis encoding (3 features per qubit)
- Compare different axis combinations
- Analyze the expressivity of different encoding approaches

## 1) The Three Rotation Axes

On the Bloch sphere, we can rotate around three axes:

| Gate | Axis | Effect |
|------|------|--------|
| R_X(θ) | X-axis | Tilts the state between |0⟩ and |1⟩ |
| R_Y(θ) | Y-axis | Rotates the state around the Y-axis |
| R_Z(θ) | Z-axis | Adds a relative phase (no population change) |

**Mathematically:**

$$R_X(\theta) = \begin{pmatrix} \cos(\theta/2) & -i\sin(\theta/2) \\ -i\sin(\theta/2) & \cos(\theta/2) \end{pmatrix}$$

$$R_Y(\theta) = \begin{pmatrix} \cos(\theta/2) & -\sin(\theta/2) \\ \sin(\theta/2) & \cos(\theta/2) \end{pmatrix}$$

$$R_Z(\theta) = \begin{pmatrix} e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2} \end{pmatrix}$$

**Important Difference:**
- $R_X$ and $R_Y$ change the **amplitudes** (measurable probabilities)
- $R_Z$ only changes the **phase** (no change in measurement results, but important for interference)

In [ ]:
# Setup
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt

# Create device
dev = qml.device("default.qubit", wires=1)

print("Setup complete! Ready for extended Angle Encoding.")

## 2) Visualization of the Three Gates

Let's visualize the effect of the three gates on the initial state |0⟩:

In [ ]:
# Visualization: Bloch sphere for different angles

def plot_bloch_sphere_angles(gate_name, angles):
    """Plot Bloch sphere with different angles for a gate."""
    fig, axes = plt.subplots(1, len(angles), figsize=(4*len(angles), 3), subplot_kw={'projection': '3d'})
    
    if len(angles) == 1:
        axes = [axes]
    
    for ax, theta in zip(axes, angles):
        # Compute qubit state
        @qml.qnode(dev)
        def circuit():
            if gate_name == 'RX':
                qml.RX(theta, wires=0)
            elif gate_name == 'RY':
                qml.RY(theta, wires=0)
            elif gate_name == 'RZ':
                qml.RZ(theta, wires=0)
            return qml.state()
        
        state = circuit()
        
        # Compute Bloch vector
        # |ψ⟩ = α|0⟩ + β|1⟩, with α = cos(θ/2), β = e^{iφ}sin(θ/2)
        alpha, beta = state[0], state[1]
        theta_bloch = 2 * np.arccos(np.abs(alpha))
        phi_bloch = np.angle(beta) - np.angle(alpha) if np.abs(alpha) > 1e-10 else 0
        
        # Cartesian coordinates
        x = np.sin(theta_bloch) * np.cos(phi_bloch)
        y = np.sin(theta_bloch) * np.sin(phi_bloch)
        z = np.cos(theta_bloch)
        
        # Draw Bloch sphere
        u = np.linspace(0, 2 * np.pi, 100)
        v = np.linspace(0, np.pi, 100)
        x_sphere = np.outer(np.cos(u), np.sin(v))
        y_sphere = np.outer(np.sin(u), np.sin(v))
        z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
        
        ax.plot_surface(x_sphere, y_sphere, z_sphere, alpha=0.1, color='blue')
        ax.plot([0, 1], [0, 0], [0, 0], 'r-', linewidth=2)  # X-axis
        ax.plot([0, 0], [0, 1], [0, 0], 'g-', linewidth=2)  # Y-axis
        ax.plot([0, 0], [0, 0], [0, 1], 'b-', linewidth=2)  # Z-axis
        
        # Draw state vector
        ax.quiver(0, 0, 0, x, y, z, color='purple', linewidth=3)
        
        ax.set_xlim([-1.2, 1.2])
        ax.set_ylim([-1.2, 1.2])
        ax.set_zlim([-1.2, 1.2])
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title(f'{gate_name}({theta:.2f})')
    
    plt.tight_layout()
    plt.show()

# Different angles for each gate
angles = [0, np.pi/4, np.pi/2, np.pi]

print("R_X rotations:")
plot_bloch_sphere_angles('RX', angles)

print("\nR_Y rotations:")
plot_bloch_sphere_angles('RY', angles)

print("\nR_Z rotations (note: only phase change, no position change!):")
plot_bloch_sphere_angles('RZ', angles)

## 3) Multi-Axis Encoding

**Idea:** Instead of using only one angle per qubit, we use all three axes!

**Advantages:**
- 3 features per qubit instead of 1 (more efficient)
- More variation possibilities in the state space
- Phase information is encoded (important for certain applications)

**Example:**
For features [x₁, y₁, z₁, x₂, y₂, z₂] with 2 qubits:

```
Qubit 0: R_X(x₁) → R_Y(y₁) → R_Z(z₁)
Qubit 1: R_X(x₂) → R_Y(y₂) → R_Z(z₂)
```

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> a) Implement Multi-Axis Encoding</h3>
  
  Implement multi-axis encoding using all three rotation axes. The function should:
  
  1. Take an array of 3×n_qubits features
  2. For each qubit, apply $R_X$, $R_Y$, and $R_Z$ rotations with the corresponding features
  3. Test with 2 qubits (6 features total)
  
  <hr>
  <i class="fas fa-wrench" style="font-size: 20px"></i> &nbsp; <strong>Variables:</strong>
  <ul>
    <li><code>features</code>: numpy array of shape (3 * n_qubits,) with values in [0, π]</li>
    <li><code>n_qubits</code>: number of qubits</li>
  </ul>
  <hr>
  
</div>

In [ ]:
# Exercise: Implement Multi-Axis Encoding

def multi_axis_encoding(features, n_qubits):
    """
    Encode features using all three rotation axes.
    
    Args:
        features: numpy array of shape (3 * n_qubits,) with values in [0, π]
                   Order: [x1, y1, z1, x2, y2, z2, ...]
        n_qubits: number of qubits
    
    Returns:
        A quantum node that performs the encoding
    """
    # YOUR CODE HERE
    # Check that features has the correct length (3 * n_qubits)
    # For each qubit: apply R_X, R_Y, R_Z with the corresponding features
    
    @qml.qnode(dev)
    def circuit():
        # TODO: Implement the Multi-Axis Encoding
        pass
    
    return circuit

# Test
test_features = np.array([0.3, 0.7, 0.5, 0.9, 0.2, 0.8])  # 2 qubits × 3 axes
test_circuit = multi_axis_encoding(test_features, n_qubits=2)
print("Multi-Axis Encoding Circuit:")
print(qml.draw(test_circuit)())

# Print state
print("\nState vector:")
print(test_circuit())

## 4) Comparing Different Axis Combinations

Which combination is best? That depends on the application!

| Combination | Features per Qubit | Advantages | Disadvantages |
|-------------|-------------------|------------|---------------|
| R_Y only | 1 | Simple, easy to understand | Limited expressivity |
| R_X + R_Y | 2 | Covers X-Y plane | No phase info |
| R_Y + R_Z | 2 | Phase + amplitudes | No X effect |
| R_X + R_Y + R_Z | 3 | Full control | More parameters |

**Tip:** For many ML applications, $R_Y$ + $R_Z$ is sufficient, since phase information is often important!

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> b) Implement Different Encoding Variants</h3>
  
  Implement two additional encoding variants and compare them with the standard $R_Y$ encoding. The functions should:
  
  1. **Variant 2** ($R_X$ + $R_Y$): Apply $R_X$ followed by $R_Y$ for each qubit
  2. **Variant 3** ($R_Y$ + $R_Z$): Apply $R_Y$ followed by $R_Z$ for each qubit
  3. Compare the resulting states for the same input features
  
  <hr>
  <i class="fas fa-wrench" style="font-size: 20px"></i> &nbsp; <strong>Variables:</strong>
  <ul>
    <li><code>features</code>: numpy array with 2×n_qubits features</li>
    <li><code>n_qubits</code>: number of qubits</li>
  </ul>
  
</div>

In [ ]:
# Exercise: Implement different encoding variants and compare them

def encoding_variant_1(features, n_qubits):
    """R_Y only (standard Angle Encoding)."""
    @qml.qnode(dev)
    def circuit():
        for i in range(n_qubits):
            qml.RY(features[i], wires=i)
        return qml.state()
    return circuit

def encoding_variant_2(features, n_qubits):
    """
    R_X + R_Y Encoding.
    features: [x1, y1, x2, y2, ...]
    """
    @qml.qnode(dev)
    def circuit():
        # YOUR CODE HERE
        # Implement R_X + R_Y Encoding
        pass
    return circuit

def encoding_variant_3(features, n_qubits):
    """
    R_Y + R_Z Encoding.
    features: [y1, z1, y2, z2, ...]
    """
    @qml.qnode(dev)
    def circuit():
        # YOUR CODE HERE
        # Implement R_Y + R_Z Encoding
        pass
    return circuit

# Test features for different variants
features_1 = np.array([0.5, 0.7])  # For R_Y only: 2 features, 2 qubits
features_2 = np.array([0.3, 0.5, 0.7, 0.9])  # For R_X+R_Y: 4 features, 2 qubits
features_3 = np.array([0.5, 0.3, 0.7, 0.8])  # For R_Y+R_Z: 4 features, 2 qubits

print("=== Variant 1: R_Y only ===")
print("Features:", features_1)
try:
    circ1 = encoding_variant_1(features_1, n_qubits=2)
    print(qml.draw(circ1)())
    print("State:", circ1())
except Exception as e:
    print(f"Error: {e}")

print("\n=== Variant 2: R_X + R_Y ===")
print("Features:", features_2)
try:
    circ2 = encoding_variant_2(features_2, n_qubits=2)
    print(qml.draw(circ2)())
    print("State:", circ2())
except Exception as e:
    print(f"Not yet implemented: {e}")

print("\n=== Variant 3: R_Y + R_Z ===")
print("Features:", features_3)
try:
    circ3 = encoding_variant_3(features_3, n_qubits=2)
    print(qml.draw(circ3)())
    print("State:", circ3())
except Exception as e:
    print(f"Not yet implemented: {e}")

## 5) Comparing Expressivity

Let's quantitatively compare the different encoding variants!

**Method:** We compute the average pairwise distance of the resulting states in Hilbert space.

<div id="exercise" class="alert alert-info">
  <h3><i class="fa fa-laptop" style="font-size:28px"></i> <br><br> c) Compute Expressivity of Encoding Variants</h3>
  
  Implement a function to compute and compare the expressivity of different encoding variants. The function should:
  
  1. Generate random features for each variant
  2. Create multiple states using each encoding
  3. Compute the average pairwise distance using the Fubini-Study metric: $d = \arccos(|\langle\psi|\phi\rangle|)$
  4. Return a dictionary with the average distances for each variant
  
  <hr>
  <i class="fas fa-wrench" style="font-size: 20px"></i> &nbsp; <strong>Variables:</strong>
  <ul>
    <li><code>n_qubits</code>: number of qubits</li>
    <li><code>n_samples</code>: number of random samples to generate</li>
  </ul>
  
</div>

In [ ]:
# Exercise: Compute the expressivity of the different encoding variants

def compute_expressivity(n_qubits, n_samples=100):
    """
    Compare the expressivity of different encoding variants.
    
    Returns:
        Dictionary with average pairwise distance for each variant
    """
    results = {}
    
    # Generate random features
    np.random.seed(42)
    
    # Variant 1: R_Y only (1 feature per qubit)
    features_1 = np.random.uniform(0, np.pi, n_qubits)
    states_1 = []
    for _ in range(n_samples):
        features_1 = np.random.uniform(0, np.pi, n_qubits)
        circ = encoding_variant_1(features_1, n_qubits)
        states_1.append(circ())
    
    # Variant 2: R_X + R_Y (2 features per qubit)
    states_2 = []
    for _ in range(n_samples):
        features_2 = np.random.uniform(0, np.pi, 2 * n_qubits)
        circ = encoding_variant_2(features_2, n_qubits)
        states_2.append(circ())
    
    # Variant 3: R_Y + R_Z (2 features per qubit)
    states_3 = []
    for _ in range(n_samples):
        features_3 = np.random.uniform(0, np.pi, 2 * n_qubits)
        circ = encoding_variant_3(features_3, n_qubits)
        states_3.append(circ())
    
    # YOUR CODE HERE
    # Compute the average pairwise distance for each variant
    # Hint: Use the Fubini-Study metric: d = arccos(|⟨ψ|φ⟩|)
    
    return results

# Test
print("Computing expressivity for 2 qubits...")
try:
    results = compute_expressivity(n_qubits=2, n_samples=50)
    print("\nResults (average distance):")
    for variant, dist in results.items():
        print(f"  {variant}: {dist:.4f}")
except Exception as e:
    print(f"Error: {e}")
    print("\nNote: First implement encoding variants 2 and 3!")

## 6) Summary

### What we learned

1. **R_X, R_Y, R_Z** rotate around different axes of the Bloch sphere
2. **R_Z** is special: It only changes the phase, not the measurement probabilities
3. **Multi-Axis Encoding** allows more features per qubit
4. The **choice of axes** depends on the application:
   - ML with amplitudes: R_X + R_Y is often sufficient
   - Phase important: R_Y + R_Z
   - Full control: All three

### When is Multi-Axis useful?

✅ **Yes**, if:
- You have many features and want to save qubits
- Phase information is important for your application
- You need maximum expressivity

❌ **No**, if:
- Simplicity is more important
- You only have few features (then R_Y is sufficient)
- The additional complexity doesn't provide an advantage

### Next Steps

- Try the encoding variants in Data Reuploading!
- Compare with Amplitude Encoding
- Experiment with the order of gates (R_X→R_Y vs R_Y→R_X)